In [15]:
from src.agent_class import Agent
from src.map_class import Map
from src.order_class import Order
from src.visualize_simulation import visualize_simulation as vs
from src.Astar_route import a_star
from IPython.display import clear_output
import random
import numpy as np
from src.Map_visualize import map_visualize
import matplotlib.pyplot as plt

In [16]:
num_node = 15
gen_prob = 0.3
deadtime = 100
current_time = 0
order_id_counter = 1
num_agent = 3
attitude = a_star
vehicle_speed = 75
agents = {}
active_orders = {}

ST_PENDING = "PENDING"
ST_ASSIGNED = "ASSIGNED"
ST_COMPLETED = "COMPLETED"

log_filename = "simulation_log_004.csv"

In [17]:
map = Map(num_node=num_node)

for i in range(num_agent):
    node_list = map.node_list
    start_node = random.choice(node_list)
    agent_id = f"AMR_{i:03d}"

    agents[i] = Agent(attitude=attitude,agent_id=agent_id,start_node=start_node,vehicle_speed=vehicle_speed)

print(f"--- エージェントが {len(agents)} 台生成されました ---")
for idx, ag in agents.items():
    print(f"Key: {idx} | ID: {ag.agent_id} | 初期位置: ノード {ag.current_node}")

--- エージェントが 3 台生成されました ---
Key: 0 | ID: AMR_000 | 初期位置: ノード 8
Key: 1 | ID: AMR_001 | 初期位置: ノード 8
Key: 2 | ID: AMR_002 | 初期位置: ノード 2


In [18]:
print("クラス連動シミュレーションを開始します... (Ctrl+C で停止)")

try:
    while True:
        map.update()
        if np.random.rand() < gen_prob:
            oid = f"ORD_{order_id_counter:05d}"

            # 関数ではなく、Order クラスの設計図から実体（インスタンス）を生成！
            new_ord = Order(
                order_id=oid,
                current_time=current_time,
                deadtime=deadtime,
                num_node=num_node,
            )
            active_orders[oid] = new_ord
            order_id_counter += 1

        # --- STEP 3: 配分マッチング（管制塔のロジック） ---
        for oid, order in active_orders.items():
            if order.status == "PENDING":  # 💡クラスなので辞書型ではなくドット記法

                best_agent = None
                min_distance = float("inf")

                # 待機中（IDLE）のロボットの中で、荷物位置（origin）に一番近い子を探す
                for ag in agents.values():
                    if ag.status == "IDLE":
                        # 改良した直線距離計算に関数を引き渡す
                        dist = get_straight_distance(
                            ag.current_node, order.origin, map
                        )
                        if dist < min_distance:
                            min_distance = dist
                            best_agent = ag

                # 一番近いロボットが見つかったら、仕事を正式に割り当てる
                if best_agent is not None:
                    # エージェントにマップクラス（sim_map）ごと仕事を渡す
                    best_agent.assign_order(order, map)
                    order.status = "ASSIGNED"  # 💡ドット記法でステータス更新
                    print(
                        f"🤖 [配分] {best_agent.agent_id} が オーダー {oid} を担当します"
                    )

        # --- STEP 4: エージェントの移動 ＆ 経路リプランニング ---
        for ag in agents.values():
            # エージェントにマップクラスを渡して1秒進める
            # この内部で「渋滞を考慮した移動時間の計算」や「交差点に着くたびのA*再計算」が自動で走ります
            ag.update(map)


except KeyboardInterrupt:
    print("\nシュミレーションを終了しました。")

クラス連動シミュレーションを開始します... (Ctrl+C で停止)


NameError: name 'get_straight_distance' is not defined

In [19]:
print(map.graph)

{'0': {'pos': (17, 89), 'edges': [('1', 39), ('8', 68)]}, '1': {'pos': (14, 50), 'edges': [('6', 66), ('5', 34)]}, '2': {'pos': (28, 7), 'edges': [('1', 45), ('10', 85)]}, '3': {'pos': (48, 71), 'edges': [('11', 42), ('7', 46)]}, '4': {'pos': (31, 57), 'edges': [('12', 22), ('2', 50)]}, '5': {'pos': (45, 34), 'edges': [('8', 25), ('7', 32)]}, '6': {'pos': (71, 85), 'edges': [('5', 57), ('11', 65)]}, '7': {'pos': (13, 41), 'edges': [('2', 37), ('11', 39)]}, '8': {'pos': (68, 44), 'edges': [('1', 54), ('0', 68)]}, '9': {'pos': (85, 82), 'edges': [('7', 82), ('6', 14)]}, '10': {'pos': (15, 91), 'edges': [('6', 56), ('14', 22)]}, '11': {'pos': (6, 80), 'edges': [('3', 42), ('6', 65)]}, '12': {'pos': (52, 50), 'edges': [('8', 17), ('3', 21)]}, '13': {'pos': (39, 30), 'edges': [('12', 23), ('2', 25)]}, '14': {'pos': (32, 77), 'edges': [('4', 20), ('8', 48)]}}


In [20]:
print(map.congestions)

{('0', '1'): 1.0, ('0', '8'): 1.0, ('1', '6'): 1.0, ('1', '5'): 1.0, ('2', '1'): 1.0, ('2', '10'): 1.0, ('3', '11'): 1.0, ('3', '7'): 1.0, ('4', '12'): 1.0, ('4', '2'): 1.0, ('5', '8'): 1.0, ('5', '7'): 1.0, ('6', '5'): 1.0, ('6', '11'): 1.0, ('7', '2'): 1.0, ('7', '11'): 1.0, ('8', '1'): 1.0, ('8', '0'): 1.0, ('9', '7'): 1.0, ('9', '6'): 1.0, ('10', '6'): 1.0, ('10', '14'): 1.0, ('11', '3'): 1.0, ('11', '6'): 1.0, ('12', '8'): 1.0, ('12', '3'): 1.0, ('13', '12'): 1.0, ('13', '2'): 1.0, ('14', '4'): 1.0, ('14', '8'): 1.0}


In [ ]:


print(agents.keys.current_node)

AttributeError: 'builtin_function_or_method' object has no attribute 'current_node'